**Assignment Linear Regression-King county dataset** by Aparna Devi Nangunoori

# Step 1: Load the dataset

In [10]:

import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from tkinter import Tk, filedialog

root = Tk()
root.withdraw()
root.attributes("-topmost", True)

file_path = filedialog.askopenfilename(
    title="Select King County Housing Dataset",
    filetypes=[("CSV files", "*.csv")]
)
root.destroy()

df = pd.read_csv(file_path, sep="\t")

print("Observations:", df.shape[0])
print("Variables:", df.shape[1])
print("Missing values:", df.isnull().sum().sum())


Observations: 22687
Variables: 22
Missing values: 0


# Step 2: Identify response and predictor variables

In [11]:
print("Dataset columns:")
print(df.columns.tolist())

print("\nResponse variable:")
print("SalePrice")

print("\nNumerical variable correlations with SalePrice:")

numeric_df = df.select_dtypes(include=["number"])

correlations = (
    numeric_df.corr()["SalePrice"]
    .sort_values(ascending=False)
)

print(correlations)


Dataset columns:
['DocumentDate', 'SalePrice', 'PropertyID', 'PropertyType', 'ym', 'zhvi_px', 'zhvi_idx', 'AdjSalePrice', 'NbrLivingUnits', 'SqFtLot', 'SqFtTotLiving', 'SqFtFinBasement', 'Bathrooms', 'Bedrooms', 'BldgGrade', 'YrBuilt', 'YrRenovated', 'TrafficNoise', 'LandVal', 'ImpsVal', 'ZipCode', 'NewConstruction']

Response variable:
SalePrice

Numerical variable correlations with SalePrice:
SalePrice          1.000000
AdjSalePrice       0.985465
ImpsVal            0.825453
LandVal            0.804961
SqFtTotLiving      0.689898
BldgGrade          0.666884
Bathrooms          0.524391
Bedrooms           0.311403
SqFtFinBasement    0.295041
SqFtLot            0.134767
YrRenovated        0.105955
zhvi_px            0.096750
zhvi_idx           0.096750
YrBuilt            0.082345
NbrLivingUnits     0.023782
TrafficNoise      -0.011710
PropertyID        -0.026166
ZipCode           -0.071767
Name: SalePrice, dtype: float64


- Interpretation: Living area and building grade have the strongest positive correlations among these property characteristics.

Although AdjSalePrice, ImpsVal, and LandVal have high correlations, we will exclude them from the model to avoid relying on adjusted prices or valuation information that may not be available when predicting a future sale.

# STEP 3: Full Multiple Linear Regression

Before applying forward stepwise regression, we need a full regression model for comparison.

We will use 11 predictors and split the data into 80% training and 20% testing.

In [12]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Define response variable
y = df["SalePrice"]

# Define predictor variables
predictors = [
    "SqFtTotLiving",
    "SqFtLot",
    "Bathrooms",
    "Bedrooms",
    "BldgGrade",
    "SqFtFinBasement",
    "YrBuilt",
    "YrRenovated",
    "NbrLivingUnits",
    "TrafficNoise",
    "NewConstruction"
]

X = df[predictors].astype(float)

# Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

# Add intercept
X_train_const = sm.add_constant(
    X_train, has_constant="add"
)
X_test_const = sm.add_constant(
    X_test, has_constant="add"
)

# Train full regression model
full_model = sm.OLS(
    y_train, X_train_const
).fit()

# Predict test data
full_predictions = full_model.predict(X_test_const)

# Evaluate model
full_r2 = r2_score(y_test, full_predictions)
full_rmse = np.sqrt(
    mean_squared_error(y_test, full_predictions)
)

print("FULL MULTIPLE LINEAR REGRESSION")
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print("\nTest R²:", round(full_r2, 4))
print("Test RMSE: $", round(full_rmse, 2))
print("Training AIC:", round(full_model.aic, 2))

print("\nRegression Summary:")
print(full_model.summary())


FULL MULTIPLE LINEAR REGRESSION
Training observations: 18149
Testing observations: 4538

Test R²: 0.5835
Test RMSE: $ 235220.26
Training AIC: 498342.38

Regression Summary:
                            OLS Regression Results                            
Dep. Variable:              SalePrice   R-squared:                       0.579
Model:                            OLS   Adj. R-squared:                  0.579
Method:                 Least Squares   F-statistic:                     2271.
Date:                Sun, 25 Oct 2026   Prob (F-statistic):               0.00
Time:                        22:11:30   Log-Likelihood:            -2.4916e+05
No. Observations:               18149   AIC:                         4.983e+05
Df Residuals:                   18137   BIC:                         4.984e+05
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                      coef    std err

Six predictors are statistically significant at the 5% level.

However, we will not manually remove the nonsignificant predictors. We will let forward stepwise regression select variables using AIC, as required.

# STEP 4: Forward Stepwise Regression using AIC

In [13]:
import statsmodels.api as sm
import pandas as pd

def forward_stepwise_selection(X, y):

    selected = []
    remaining = list(X.columns)

    # Start with intercept-only model
    intercept_only = pd.DataFrame(index=X.index)

    current_model = sm.OLS(
        y,
        sm.add_constant(
            intercept_only,
            has_constant="add"
        )
    ).fit()

    current_aic = current_model.aic

    print("Initial AIC:", round(current_aic, 2))
    print("\nForward Stepwise Selection:\n")

    step = 1

    while remaining:

        results = []

        for candidate in remaining:

            features = selected + [candidate]

            model = sm.OLS(
                y,
                sm.add_constant(
                    X[features],
                    has_constant="add"
                )
            ).fit()

            results.append(
                (model.aic, candidate)
            )

        best_aic, best_candidate = min(results)

        if best_aic < current_aic:

            selected.append(best_candidate)
            remaining.remove(best_candidate)
            current_aic = best_aic

            print(
                f"Step {step}: "
                f"Added {best_candidate} | "
                f"AIC = {current_aic:.2f}"
            )

            step += 1

        else:
            break

    return selected


# Run selection on training data only
selected_features = forward_stepwise_selection(
    X_train,
    y_train
)

print("\nFINAL SELECTED PREDICTORS:")
print(selected_features)

print("\nNumber of selected predictors:",
      len(selected_features))


Initial AIC: 514037.48

Forward Stepwise Selection:

Step 1: Added SqFtTotLiving | AIC = 502466.87
Step 2: Added BldgGrade | AIC = 500764.71
Step 3: Added YrBuilt | AIC = 498800.59
Step 4: Added Bedrooms | AIC = 498467.64
Step 5: Added Bathrooms | AIC = 498343.20
Step 6: Added NewConstruction | AIC = 498336.33
Step 7: Added SqFtFinBasement | AIC = 498335.90

FINAL SELECTED PREDICTORS:
['SqFtTotLiving', 'BldgGrade', 'YrBuilt', 'Bedrooms', 'Bathrooms', 'NewConstruction', 'SqFtFinBasement']

Number of selected predictors: 7


The AIC decreased from **514,037.48** to **498,335.90**.

This indicates that the selected predictors improve the model's balance between fit and complexity.

This selection does not necessarily mean every selected variable has a p-value below 0.05.

# STEP 5: Train and Evaluate Stepwise Regression

In [14]:
# Prepare selected predictors
X_train_step = sm.add_constant(
    X_train[selected_features],
    has_constant="add"
)

X_test_step = sm.add_constant(
    X_test[selected_features],
    has_constant="add"
)

# Train stepwise regression model
stepwise_model = sm.OLS(
    y_train,
    X_train_step
).fit()

# Predict house prices on test data
stepwise_predictions = stepwise_model.predict(
    X_test_step
)

# Evaluate model
stepwise_r2 = r2_score(
    y_test,
    stepwise_predictions
)

stepwise_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        stepwise_predictions
    )
)

print("STEPWISE REGRESSION RESULTS")
print("---------------------------")

print("Selected predictors:", selected_features)
print("Number of predictors:", len(selected_features))

print("\nTest R²:", round(stepwise_r2, 4))
print("Test RMSE: $", round(stepwise_rmse, 2))
print("Training AIC:", round(stepwise_model.aic, 2))

print("\nStepwise Regression Summary:")
print(stepwise_model.summary())


STEPWISE REGRESSION RESULTS
---------------------------
Selected predictors: ['SqFtTotLiving', 'BldgGrade', 'YrBuilt', 'Bedrooms', 'Bathrooms', 'NewConstruction', 'SqFtFinBasement']
Number of predictors: 7

Test R²: 0.5835
Test RMSE: $ 235212.7
Training AIC: 498335.9

Stepwise Regression Summary:
                            OLS Regression Results                            
Dep. Variable:              SalePrice   R-squared:                       0.579
Model:                            OLS   Adj. R-squared:                  0.579
Method:                 Least Squares   F-statistic:                     3569.
Date:                Sun, 25 Oct 2026   Prob (F-statistic):               0.00
Time:                        22:11:43   Log-Likelihood:            -2.4916e+05
No. Observations:               18149   AIC:                         4.983e+05
Df Residuals:                   18141   BIC:                         4.984e+05
Df Model:                           7                                 

Interpretation: The stepwise model explains approximately **58.35%** of the variation in test-set house prices. Its RMSE is **$7.56** lower than the full model, which is a negligible improvement.

The stepwise model is therefore preferable so far because it uses fewer variables without sacrificing predictive performance.

One observation from your regression summary: SqFtFinBasement has a p-value of 0.119. Although it is not statistically significant at 0.05, AIC-based selection retained it because it slightly improved the model's AIC.

# STEP 6: Three-Variable Multiple Linear Regression

We will select these three variables based on their strong correlations with house prices:
1. SqFtTotLiving — Living area
2. BldgGrade — Building quality grade
3. Bathrooms — Number of bathrooms

In [15]:
three_features = [
    "SqFtTotLiving",
    "BldgGrade",
    "Bathrooms"
]

# Prepare training and testing data
X_train_3 = sm.add_constant(
    X_train[three_features],
    has_constant="add"
)

X_test_3 = sm.add_constant(
    X_test[three_features],
    has_constant="add"
)

# Train model
three_model = sm.OLS(
    y_train,
    X_train_3
).fit()

# Predict test prices
three_predictions = three_model.predict(
    X_test_3
)

# Calculate model performance
three_r2 = r2_score(
    y_test,
    three_predictions
)

three_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        three_predictions
    )
)

print("THREE-VARIABLE REGRESSION RESULTS")
print("---------------------------------")

print("Predictors:", three_features)
print("Number of predictors:", len(three_features))

print("\nTest R²:", round(three_r2, 4))
print("Test RMSE: $", round(three_rmse, 2))
print("Training AIC:", round(three_model.aic, 2))

print("\nRegression Summary:")
print(three_model.summary())


THREE-VARIABLE REGRESSION RESULTS
---------------------------------
Predictors: ['SqFtTotLiving', 'BldgGrade', 'Bathrooms']
Number of predictors: 3

Test R²: 0.529
Test RMSE: $ 250132.26
Training AIC: 500694.51

Regression Summary:
                            OLS Regression Results                            
Dep. Variable:              SalePrice   R-squared:                       0.521
Model:                            OLS   Adj. R-squared:                  0.521
Method:                 Least Squares   F-statistic:                     6572.
Date:                Sun, 25 Oct 2026   Prob (F-statistic):               0.00
Time:                        22:11:46   Log-Likelihood:            -2.5034e+05
No. Observations:               18149   AIC:                         5.007e+05
Df Residuals:                   18145   BIC:                         5.007e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                    

The model explains approximately **52.9%** of the variation in test-set house prices.

Compared with the stepwise model, the three-variable model has a **higher RMSE** and **lower R²**. Therefore, reducing the number of predictors to three results in a **noticeable loss of predictive accuracy**.

# STEP 7: Compare Full, Stepwise and Three-Variable Models

In [16]:
comparison = pd.DataFrame({
    "Model": [
        "Full Regression",
        "Stepwise Regression",
        "Three-Variable Regression"
    ],
    "Predictors": [
        len(predictors),
        len(selected_features),
        len(three_features)
    ],
    "Test R²": [
        full_r2,
        stepwise_r2,
        three_r2
    ],
    "Test RMSE ($)": [
        full_rmse,
        stepwise_rmse,
        three_rmse
    ],
    "Training AIC": [
        full_model.aic,
        stepwise_model.aic,
        three_model.aic
    ]
})

print("FINAL MODEL COMPARISON")
display(comparison.round(4))

# Identify model with lowest test RMSE
best_model = comparison.loc[
    comparison["Test RMSE ($)"].idxmin()
]

print("\nModel with lowest test RMSE:")
print(best_model["Model"])

print("\nRMSE difference: Three-Variable vs Stepwise")
print(
    f"${three_rmse - stepwise_rmse:,.2f}"
)


FINAL MODEL COMPARISON


,Model,Predictors,Test R²,Test RMSE ($),Training AIC
0,Full Regression,11,0.5835,235220.2564,498342.3799
1,Stepwise Regression,7,0.5835,235212.6975,498335.9006
2,Three-Variable Regression,3,0.5290,250132.2597,500694.5076



Model with lowest test RMSE:
Stepwise Regression

RMSE difference: Three-Variable vs Stepwise
$14,919.56


**Interpretation**

I would choose the **seven-predictor stepwise model** because it achieves nearly identical performance to the full model using four fewer variables. I would not choose the three-variable model because its RMSE is approximately **$14,920 higher.**

# STEP 8: Predict Price of an Unseen House

In [17]:
# Example house characteristics
new_house = pd.DataFrame([{
    "SqFtTotLiving": 2000,
    "BldgGrade": 7,
    "YrBuilt": 2000,
    "Bedrooms": 3,
    "Bathrooms": 2,
    "NewConstruction": 0,
    "SqFtFinBasement": 0
}])

# Use predictors selected by stepwise regression
new_house_X = sm.add_constant(
    new_house[selected_features],
    has_constant="add"
)

# Predict using stepwise model
predicted_price = stepwise_model.predict(new_house_X)

print("UNSEEN HOUSE PRICE PREDICTION")
print("-----------------------------")

print("House characteristics:")
display(new_house)

print(
    f"\nPredicted Sale Price: "
    f"${predicted_price.iloc[0]:,.2f}"
)


UNSEEN HOUSE PRICE PREDICTION
-----------------------------
House characteristics:


,SqFtTotLiving,BldgGrade,YrBuilt,Bedrooms,Bathrooms,NewConstruction,SqFtFinBasement
0,2000,7,2000,3,2,0,0



Predicted Sale Price: $329,631.92


**Question:**
Would you choose a simpler model with nearly identical performance? Why or why not?

Yes. I would choose a simpler model if it provides nearly identical predictive performance to a more complex model.

In this analysis, the stepwise regression model uses seven predictors compared with eleven in the full regression model, while achieving almost identical R² and RMSE values.

A simpler model offers several advantages, including easier interpretation, reduced data requirements, lower maintenance effort, and potentially less overfitting.

However, I would not choose the three-variable model because its predictive accuracy decreases noticeably.

Therefore, I would select the seven-predictor stepwise regression model, as it provides a good balance between simplicity and predictive accuracy.

**Final Conclusion**
  
The analysis demonstrated that living area, building grade, and other property characteristics are useful predictors of house prices.

Forward stepwise regression reduced the number of predictors from eleven to seven without a meaningful loss in predictive performance.

The selected stepwise model achieved an **R²** of **0.5835** and **RMSE** of **$235,212.70**. It was also used to predict a sale price of **$329,631.92** for an example unseen house with 2,000 square feet of living area, three bedrooms, two bathrooms, and building grade 7.

The **stepwise regression model** is the preferred model for this assignment, although further validation and improvements would be required before relying on it for high-stakes business valuations.